In [0]:
# bootstrap_server='pkc-xrnwx.asia-south2.gcp.confluent.cloud:9092'
# api_key='2N2J6MCHQVCEY5VP'
# api_secret='cfltey8O7CKviHeiH7OhYD7OpOcpmRmhZlUCYJg0UX7zexbg5fIdgZx6hQfY4m9g'
# topic='credit_card_transactions'

In [0]:

# print(api_key)
# print(kafka)

In [0]:
jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

In [0]:
sample_batch=(spark.read.format('kafka')
              .option('kafka.bootstrap.servers',bootstrap_server)
              .option('subscribe',topic)
              .option('kafka.sasl.mechanism','PLAIN')
              .option('kafka.security.protocol','SASL_SSL')
              .option('kafka.sasl.jaas.config',jaas_config)
              .option('startingoffsets','earliest')
              .load()
             
              )

In [0]:
from pyspark.sql import functions as F
parsed_batch=sample_batch.select(
    F.col("key").cast("string"),
    F.col("value").cast("string"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp"),
    F.col("timestampType")
)
# display(parsed_batch)

In [0]:
parsed_batch.write.saveAsTable("city_bank.bronze.transaction_batch_test")

In [0]:
stream_df=sample_batch=(spark.readStream.format('kafka')
              .option('kafka.bootstrap.servers',bootstrap_server)
              .option('subscribe',topic)
              .option('kafka.sasl.mechanism','PLAIN')
              .option('kafka.security.protocol','SASL_SSL')
              .option('kafka.sasl.jaas.config',jaas_config)
              .option('startingoffsets','earliest')
              .load()
             
              )

In [0]:
from pyspark.sql import functions as F
parsed_streaming_df=stream_df.select(
    F.col("key").cast("string"),
    F.col("value").cast("string"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp"),
    F.col("timestampType")
)

In [0]:
streamming_query=(parsed_streaming_df.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/city_bank/source/transactions/checkpoint/").trigger(availableNow=True).toTable("city_bank.bronze.transaction_streaming_test"))
# print("query_id",streamming_query.id)

In [0]:
%sql
select * from city_bank.bronze.transaction_streaming_test;